In [1]:
# Install compatible CUDA stack
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0
!pip install -q --no-cache-dir \
  bitsandbytes==0.45.5 triton==3.2.0 \
  transformers==4.46.3 accelerate==0.34.2 \
  sentence-transformers==3.0.1 faiss-cpu==1.8.0.post1 \
  pandas==2.2.2 numpy==1.26.4 scipy==1.11.4 tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.8.0+cu128 requires torch==2.8.0, but you have torch 2.6.0+cu124 which is incompatible.
torchvision 0.23.0+cu128 requires torch==2.8.0, but you have torch 2.6.0+cu124 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.23.0+cu128 requires torch==2.8.0, but you have torch 2.6.0+cu124 which is incompatible.


In [ ]:
import os, signal
os.kill(os.getpid(), signal.SIGKILL)

In [4]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.23.0+cu128
Uninstalling torchvision-0.23.0+cu128:
  Successfully uninstalled torchvision-0.23.0+cu128


In [1]:
import json
import time
import random
from pathlib import Path
from datetime import datetime, UTC

import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [13]:
# ---- Generation settings ----
MAX_NEW_TOKENS = 80
TEMPERATURE = 0.0
TOP_K_RAG = 6
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SLEEP_SEC = 0.0
SEED = 42

# ---- DP prompt control (None = full book) ----
DP_MAX_BOOK_CHARS = None  # set e.g. 30000 if you want to cap DP context

# ---- Base folder in your Drive (edit once) ----
BASE = Path("/workspace")

# Use your three book.txt + QA CSV files here
JOBS = [
    {
        "name": "news",
        "book_txt": BASE / "news_book.txt",
        "qa_csv":   BASE / "news_news_llm_retry_df_qa.combined_story_minerva_balanced_150.csv",
    },
    {
        "name": "scifi",
        "book_txt": BASE / "scifi_book.txt",
        "qa_csv":   BASE / "scifi_20260312T170655Z_df_qa.combined_story_minerva_balanced_150.csv",
    },
    {
        "name": "thriller",
        "book_txt": BASE / "book.txt",
        "qa_csv":   BASE / "thriller_20260311T152054Z_df_qa.combined_story_minerva_balanced_150.csv",
    },
]

OUT_DIR = BASE / "outputs_phi3_mini_4k_dp_rag"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def ts():
    return datetime.now(UTC).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def log_line(msg, log_path=None):
    line = f"[{ts()}] {msg}"
    print(line, flush=True)
    if log_path is not None:
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")

def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def append_jsonl(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def row_key(row: pd.Series, i: int):
    if "combined_q_idx" in row and pd.notna(row["combined_q_idx"]):
        return f"combined_q_idx:{int(row['combined_q_idx'])}"
    return f"row_idx:{int(i)}"

def split_book_into_chunks(book_text: str):
    chunks = []
    chapters = book_text.split("Chapter ")
    if len(chapters) > 1:
        for ci, ch in enumerate(chapters[1:], start=1):
            paras = [p.strip() for p in ch.split("\n\n") if p.strip()]
            for pi, p in enumerate(paras, start=1):
                chunks.append({
                    "chunk_id": len(chunks),
                    "label": f"Chapter {ci}, Paragraph {pi}",
                    "text": p
                })
    else:
        paras = [p.strip() for p in book_text.split("\n\n") if p.strip()]
        for pi, p in enumerate(paras, start=1):
            chunks.append({
                "chunk_id": len(chunks),
                "label": f"Paragraph {pi}",
                "text": p
            })
    return chunks

def build_rag_index(book_text: str, embedder: SentenceTransformer):
    chunks = split_book_into_chunks(book_text)
    X = embedder.encode([c["text"] for c in chunks], convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    idx = faiss.IndexFlatL2(X.shape[1])
    idx.add(X)
    return idx, chunks

def retrieve_chunks(question: str, idx, chunks, embedder, top_k: int):
    qv = embedder.encode([question], convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    D, I = idx.search(qv, top_k)
    out = []
    for rank, (d, j) in enumerate(zip(D[0], I[0]), start=1):
        c = chunks[int(j)]
        out.append({
            "rank": rank,
            "distance": float(d),
            "chunk_id": c["chunk_id"],
            "label": c["label"],
            "text": c["text"],
        })
    return out

def build_rag_prompt(question: str, retrieved):
    context = "\n\n".join([f"[{r['label']}]\n{r['text']}" for r in retrieved])
    return f"""# Episodic Memory Benchmark
You are participating in an episodic memory test based on retrieved text.
Answer using only the retrieved chunks. If unsure, say unsure.

## Retrieved Relevant Chunks:
{context}

## Question:
{question}

Return only the final answer.
"""

def build_dp_prompt(question: str, book_text: str, max_chars=None):
    src = book_text if (max_chars is None or max_chars <= 0) else book_text[:max_chars]
    return f"""# Episodic Memory Benchmark
You are participating in an episodic memory test.
Answer using only the book text. If unsure, say unsure.

## Book Text:
{src}

## Question:
{question}

Return only the final answer.
"""

@torch.inference_mode()
def generate_answer(model, tokenizer, user_prompt, max_new_tokens=80, temperature=0.0):
    messages = [
        {"role": "system", "content": "You are an expert in memory tests."},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    if temperature == 0.0:
        do_sample = False
    else:
        do_sample = True

    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=max(temperature, 1e-5),
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0, input_ids.shape[-1]:]
    txt = tokenizer.decode(gen, skip_special_tokens=True).strip()
    return txt, int(input_ids.shape[-1])

In [4]:
!pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 7.3 MB/s  0:00:00m eta 0:00:01


In [5]:
# 1) hard cleanup first
import gc, torch
for name in ["model", "tokenizer", "embedder"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

free_gb = torch.cuda.mem_get_info()[0] / 1024**3
total_gb = torch.cuda.mem_get_info()[1] / 1024**3
print(f"GPU free: {free_gb:.2f} / {total_gb:.2f} GB")

GPU free: 23.28 / 23.55 GB


In [6]:
# 2) load Phi-3 forcing all on GPU (no auto spill), safer attention mode
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},                 # force full GPU
    torch_dtype=torch.float16,
    attn_implementation="eager",        # avoids flash-attn/window warnings
    trust_remote_code=True
)
model.eval()

print("Phi-3 loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi-3 loaded.


In [10]:
# 1) delete model objects
import gc, torch
for v in ["model", "tokenizer", "embedder"]:
    if v in globals():
        del globals()[v]

gc.collect()
torch.cuda.empty_cache()

# optional: show memory
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1024**3:.2f} / {total/1024**3:.2f} GB")

GPU free: 0.01 / 23.55 GB


In [9]:
embedder = SentenceTransformer(EMBED_MODEL, device="cpu")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,   # <- use this
    trust_remote_code=True
)
model.eval()

embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
print("Model + embedder loaded.")

NameError: name 'random' is not defined

In [7]:
def run_job(job, mode="rag", limit=None):
    assert mode in {"rag", "dp"}

    name = job["name"]
    book_txt = Path(job["book_txt"])
    qa_csv = Path(job["qa_csv"])

    assert book_txt.exists(), f"Missing: {book_txt}"
    assert qa_csv.exists(), f"Missing: {qa_csv}"

    out_jsonl = OUT_DIR / f"{name}_phi3-mini-4k_{mode}.answers.jsonl"
    run_log = OUT_DIR / f"{name}_phi3-mini-4k_{mode}.run.log"

    log_line(f"run_start mode={mode} name={name}", run_log)
    log_line(f"book_txt={book_txt}", run_log)
    log_line(f"qa_csv={qa_csv}", run_log)
    log_line(f"output={out_jsonl}", run_log)

    book_text = book_txt.read_text(encoding="utf-8", errors="replace")
    df = pd.read_csv(qa_csv)
    if limit is not None:
        df = df.head(limit).copy()

    done = {r.get("row_key") for r in read_jsonl(out_jsonl) if r.get("status") == "ok"}
    log_line(f"resume_done={len(done)} rows_total={len(df)}", run_log)

    idx, chunks = (None, None)
    if mode == "rag":
        log_line(f"building_rag_index embed_model={EMBED_MODEL}", run_log)
        idx, chunks = build_rag_index(book_text, embedder)
        log_line(f"chunks_indexed={len(chunks)}", run_log)

    for i in tqdm(range(len(df))):
        row = df.iloc[i]
        rk = row_key(row, i)
        if rk in done:
            continue

        q = str(row["question"])
        q_idx = int(row["q_idx"]) if ("q_idx" in row and pd.notna(row["q_idx"])) else int(i)
        combined_q_idx = int(row["combined_q_idx"]) if ("combined_q_idx" in row and pd.notna(row["combined_q_idx"])) else None

        retrieved = None
        if mode == "rag":
            retrieved = retrieve_chunks(q, idx, chunks, embedder, TOP_K_RAG)
            prompt = build_rag_prompt(q, retrieved)
        else:
            prompt = build_dp_prompt(q, book_text, max_chars=DP_MAX_BOOK_CHARS)

        status = "ok"
        err = None
        answer = ""
        prompt_tokens = None

        try:
            answer, prompt_tokens = generate_answer(
                model=model,
                tokenizer=tokenizer,
                user_prompt=prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
            )
        except Exception as e:
            status = "error"
            err = f"{type(e).__name__}: {e}"

        rec = {
            "timestamp_utc": ts(),
            "status": status,
            "error": err,
            "mode": mode,
            "model_id": MODEL_ID,
            "row_key": rk,
            "row_idx": int(i),
            "q_idx": q_idx,
            "combined_q_idx": combined_q_idx,
            "question": q,
            "retrieval_type": str(row["retrieval_type"]) if "retrieval_type" in row and pd.notna(row["retrieval_type"]) else None,
            "get": str(row["get"]) if "get" in row and pd.notna(row["get"]) else None,
            "correct_answer": str(row["correct_answer"]) if "correct_answer" in row and pd.notna(row["correct_answer"]) else None,
            "prompt_tokens": prompt_tokens,
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "answer": answer,
            "top_k": TOP_K_RAG if mode == "rag" else None,
            "retrieved_chunks": retrieved if mode == "rag" else None,
        }
        append_jsonl(out_jsonl, rec)

        if status == "ok":
            done.add(rk)

        if SLEEP_SEC > 0:
            time.sleep(SLEEP_SEC)

    log_line(f"run_done mode={mode} name={name} done_ok={len(done)}", run_log)

In [12]:
for job in JOBS:
    run_job(job, mode="rag", limit=None)

[2026-04-02T17:06:55Z] run_start mode=rag name=news
[2026-04-02T17:06:55Z] book_txt=/workspace/news_book.txt
[2026-04-02T17:06:55Z] qa_csv=/workspace/news_news_llm_retry_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:06:55Z] output=/workspace/outputs_phi3_mini_4k_dp_rag/news_phi3-mini-4k_rag.answers.jsonl
[2026-04-02T17:06:55Z] resume_done=0 rows_total=150
[2026-04-02T17:06:55Z] building_rag_index embed_model=sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

[2026-04-02T17:06:57Z] chunks_indexed=405


  0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


[2026-04-02T17:11:22Z] run_done mode=rag name=news done_ok=150


In [14]:
for job in JOBS:
    run_job(job, mode="dp", limit=None)

[2026-04-02T17:12:23Z] run_start mode=dp name=news
[2026-04-02T17:12:23Z] book_txt=/workspace/news_book.txt
[2026-04-02T17:12:23Z] qa_csv=/workspace/news_news_llm_retry_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:12:23Z] output=/workspace/outputs_phi3_mini_4k_dp_rag/news_phi3-mini-4k_dp.answers.jsonl
[2026-04-02T17:12:23Z] resume_done=0 rows_total=150


  0%|          | 0/150 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (21556 > 4096). Running this sequence through the model will result in indexing errors
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


[2026-04-02T17:12:47Z] run_done mode=dp name=news done_ok=0
[2026-04-02T17:12:47Z] run_start mode=dp name=scifi
[2026-04-02T17:12:47Z] book_txt=/workspace/scifi_book.txt
[2026-04-02T17:12:47Z] qa_csv=/workspace/scifi_20260312T170655Z_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:12:47Z] output=/workspace/outputs_phi3_mini_4k_dp_rag/scifi_phi3-mini-4k_dp.answers.jsonl
[2026-04-02T17:12:47Z] resume_done=0 rows_total=150


  0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


[2026-04-02T17:13:13Z] run_done mode=dp name=scifi done_ok=0
[2026-04-02T17:13:13Z] run_start mode=dp name=thriller
[2026-04-02T17:13:13Z] book_txt=/workspace/book.txt
[2026-04-02T17:13:13Z] qa_csv=/workspace/thriller_20260311T152054Z_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:13:13Z] output=/workspace/outputs_phi3_mini_4k_dp_rag/thriller_phi3-mini-4k_dp.answers.jsonl
[2026-04-02T17:13:13Z] resume_done=0 rows_total=150


  0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


[2026-04-02T17:13:52Z] run_done mode=dp name=thriller done_ok=0
